# Clasificación de Sentimiento con Transformer + fastText (Español)

## 1. Preparación
Descarga fastText en español (300 dimensiones) desde:
👉 https://fasttext.cc/docs/en/crawl-vectors.html

Archivo: `cc.es.300.vec`

Colócalo en la misma carpeta que este notebook.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import gensim

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", DEVICE)

## 2. Cargar embeddings fastText (Español)

In [ ]:
path = "cc.es.300.vec"  # Asegúrate que esté en la carpeta del notebook

print("Cargando modelo fastText (puede tardar unos minutos)...")
ft_model = gensim.models.KeyedVectors.load_word2vec_format(path)
print("Embeddings cargados. Vocab size:", len(ft_model.index_to_key))

## 3. Tokenizer con `<PAD>` y `<UNK>`

In [ ]:
class FastTextTokenizer:
    def __init__(self, ft_model):
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}
        self.idx2word = ["<PAD>", "<UNK>"]
        self.embedding_matrix = [np.zeros(ft_model.vector_size), np.random.normal(scale=0.6, size=(ft_model.vector_size,))]

        for word in ft_model.index_to_key:
            self.word2idx[word] = len(self.idx2word)
            self.idx2word.append(word)
            self.embedding_matrix.append(ft_model[word])

        self.embedding_matrix = np.array(self.embedding_matrix)

    def encode(self, text, max_len=12):
        tokens = text.lower().split()
        idxs = [self.word2idx.get(tok, 1) for tok in tokens]
        if len(idxs) < max_len:
            idxs += [0] * (max_len - len(idxs))
        return idxs[:max_len]

tokenizer = FastTextTokenizer(ft_model)
print("Tamaño vocabulario:", len(tokenizer.idx2word))

## 4. Dataset de ejemplo en español

In [ ]:
sentences = [
    ("Me encantó esta película, fue maravillosa", 1),
    ("La trama era aburrida y muy mala", 0),
    ("Una experiencia fantástica, la volvería a ver", 1),
    ("La peor película que he visto", 0)
]

class SentimentDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, label = self.data[idx]
        x = torch.tensor(self.tokenizer.encode(text), dtype=torch.long)
        y = torch.tensor(label, dtype=torch.long)
        return x, y

train_dataset = SentimentDataset(sentences, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

## 5. Modelo Transformer con embeddings fastText

In [ ]:
class TransformerClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=128, nhead=2, num_layers=2, num_classes=2):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(torch.tensor(embedding_matrix, dtype=torch.float32), freeze=False)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=nhead, dim_feedforward=hidden_dim)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        emb = emb.permute(1,0,2)
        out = self.transformer(emb)
        out = out.mean(dim=0)
        return self.fc(out)

model = TransformerClassifier(tokenizer.embedding_matrix).to(DEVICE)

## 6. Entrenamiento

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

## 7. Prueba con frases nuevas en español

In [ ]:
test_sentences = [
    "Me encantó esta película, la recomiendo mucho",
    "Fue muy aburrida y una pérdida de tiempo",
    "Estuvo bien, no fue excelente pero tampoco terrible"
]

model.eval()
with torch.no_grad():
    for s in test_sentences:
        x = torch.tensor(tokenizer.encode(s), dtype=torch.long).unsqueeze(0).to(DEVICE)
        pred = torch.argmax(model(x), dim=1).item()
        label = "Positivo" if pred == 1 else "Negativo"
        print(f"{s} -> {label}")